<a href="https://colab.research.google.com/github/Abiads/python-10-days-XebAppi/blob/main/DAY6_PYTHON.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Daily Report

**Date:** 26 March 2026

## Work Done

- Studied **Advanced File Handling** using CSV and JSON to manage structured data efficiently
- Implemented **File Operations** using `os` and `shutil` for directory and file management
- Used **List and Dictionary Comprehensions** to optimize code and improve data processing speed
- Applied **Lambda Functions** with `map()` and `filter()` for short and quick functional logic
- Explored **Advanced OOP Concepts** like Class Methods and Abstract Classes for better design structure
- Learned **Design Patterns** such as Singleton and Factory to build reusable and scalable system design
- Implemented **Concurrency** using `threading` and `multiprocessing` to improve execution performance

## Key Learnings & Observations

- **Advanced File Handling** is essential for managing real-world data formats like CSV and JSON, commonly used in APIs.
- **Comprehensions** significantly reduce code size while improving readability, saving valuable time in real projects.
- **Lambda Functions** are very useful for small, one-time logic where creating a full function is unnecessary.
- **Design Patterns** (Singleton, Factory) greatly improve code structure, making projects more scalable and easier to maintain.
- **Concurrency** improves performance by running multiple tasks simultaneously — critical in backend systems.
- **Multiprocessing** is more effective for CPU-bound tasks, while **Threading** works well for I/O-bound tasks.

---

**Prepared by:** Abhay Gupta

In [ ]:
import os
import shutil
import csv
import json
import threading
import multiprocessing
import time
from abc import ABC, abstractmethod

# --- 1. Singleton Pattern for Configuration Management ---
# The ConfigManager ensures that only one instance of the configuration exists
# throughout the application, preventing inconsistencies.
class ConfigManager:
    _instance = None
    _initialized = False

    def __new__(cls):
        if not cls._instance:
            cls._instance = super(ConfigManager, cls).__new__(cls)
        return cls._instance

    def __init__(self, config_file='app_config.json'):
        if not ConfigManager._initialized:
            self.config = {}
            self.config_file = config_file
            try:
                with open(self.config_file, 'r') as f:
                    self.config = json.load(f)
                print(f"ConfigManager: Loaded configuration from {self.config_file}")
            except FileNotFoundError:
                print(f"ConfigManager: Configuration file {self.config_file} not found. Using default/empty config.")
            ConfigManager._initialized = True

    def get_setting(self, key, default=None):
        return self.config.get(key, default)

    def set_setting(self, key, value):
        self.config[key] = value
        print(f"ConfigManager: Set setting '{key}' to '{value}'")

    def save_config(self):
        try:
            with open(self.config_file, 'w') as f:
                json.dump(self.config, f, indent=4)
            print(f"ConfigManager: Saved configuration to {self.config_file}")
        except IOError as e:
            print(f"ConfigManager: Error saving config: {e}")

# Create a sample config.json file for demonstration
sample_config_data = {
    "output_dir": "processed_data_reports",
    "raw_data_archive_dir": "raw_data_archive",
    "unit_conversion": {"celsius_to_fahrenheit": "lambda c: (c * 9/5) + 32"} # Stored as string, will be eval'd
}
with open('app_config.json', 'w') as f:
    json.dump(sample_config_data, f, indent=4)

# Initialize Configuration Manager (Singleton in action)
config_manager = ConfigManager()
OUTPUT_DIR = config_manager.get_setting("output_dir")
ARCHIVE_DIR = config_manager.get_setting("raw_data_archive_dir")
celsius_to_fahrenheit_lambda_str = config_manager.get_setting("unit_conversion").get("celsius_to_fahrenheit")
celsius_to_fahrenheit_lambda = eval(celsius_to_fahrenheit_lambda_str) # Using eval for lambda function

# --- 2. File Operations (os and shutil) ---
# These operations manage the directory structure for raw and processed data.
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(ARCHIVE_DIR, exist_ok=True)
print(f"Ensured directories '{OUTPUT_DIR}' and '{ARCHIVE_DIR}' exist.")

# --- 3. Advanced OOP: Abstract Classes and Class Methods ---
# BaseSensorProcessor is an abstract class defining a common interface for sensor processing.
# Concrete processors (like TemperatureProcessor) must implement the abstract methods.
class BaseSensorProcessor(ABC):
    def __init__(self, sensor_id, unit="C"):
        self.sensor_id = sensor_id
        self.unit = unit
        self.processed_readings = []

    @abstractmethod
    def process_reading(self, raw_reading):
        """Abstract method to process a single raw sensor reading."""
        pass

    @classmethod
    def from_config(cls, sensor_config):
        """Class method: Alternative constructor to create processor from config data."""
        sensor_id = sensor_config.get("id")
        unit = sensor_config.get("unit", "C")
        return cls(sensor_id, unit)

    def generate_report(self):
        """Generates a simple JSON report of processed data using Advanced File Handling."""
        report_data = {
            "sensor_id": self.sensor_id,
            "unit": self.unit,
            "processed_count": len(self.processed_readings),
            "average_value": sum(self.processed_readings) / len(self.processed_readings) if self.processed_readings else 0,
            "readings": self.processed_readings
        }
        report_file = os.path.join(OUTPUT_DIR, f"{self.sensor_id}_report.json")
        with open(report_file, 'w') as f:
            json.dump(report_data, f, indent=4) # Advanced File Handling: JSON
        print(f"Report for sensor {self.sensor_id} saved to {report_file}")

class TemperatureProcessor(BaseSensorProcessor):
    def __init__(self, sensor_id, unit="C"):
        super().__init__(sensor_id, unit)
        print(f"TemperatureProcessor for {sensor_id} created, unit: {unit}")

    def process_reading(self, raw_reading):
        # Example of applying a lambda function for unit conversion
        if self.unit == "F":
            # Assuming raw_reading is in Celsius, convert to Fahrenheit using the lambda
            processed_value = celsius_to_fahrenheit_lambda(float(raw_reading))
        else:
            processed_value = float(raw_reading)
        self.processed_readings.append(processed_value)
        return processed_value

class HumidityProcessor(BaseSensorProcessor):
    def __init__(self, sensor_id, unit="RH"):
        super().__init__(sensor_id, unit)
        print(f"HumidityProcessor for {sensor_id} created, unit: {unit}")

    def process_reading(self, raw_reading):
        processed_value = float(raw_reading) # No complex conversion for humidity in this example
        self.processed_readings.append(processed_value)
        return processed_value

# --- 4. Design Pattern: Factory Pattern ---
# The SensorProcessorFactory abstracts the creation of sensor processor objects.
class SensorProcessorFactory:
    def create_processor(self, sensor_type, sensor_id, unit=None):
        if sensor_type == "temperature":
            return TemperatureProcessor(sensor_id, unit or "C")
        elif sensor_type == "humidity":
            return HumidityProcessor(sensor_id, unit or "RH")
        else:
            raise ValueError(f"Unknown sensor type: {sensor_type}")

# --- Sample Raw CSV Data Generation ---
# Generates dummy CSV files to simulate incoming sensor data (Advanced File Handling: CSV)
def generate_sample_csv_data(filename, sensor_type, sensor_id, count=10):
    with open(filename, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['timestamp', 'sensor_id', 'sensor_type', 'value'])
        for i in range(count):
            timestamp = time.time() - (count - i) # Simulate timestamps
            value = round(20 + i * 0.5 if sensor_type == 'temperature' else 60 + i * 0.2, 2)
            writer.writerow([timestamp, sensor_id, sensor_type, value])
    print(f"Generated sample CSV data: {filename}")

# Generate a few sample sensor data files
raw_data_files = []
generate_sample_csv_data("sensor_temp_001.csv", "temperature", "temp_001", 15)
raw_data_files.append("sensor_temp_001.csv")
generate_sample_csv_data("sensor_humid_002.csv", "humidity", "humid_002", 12)
raw_data_files.append("sensor_humid_002.csv")
generate_sample_csv_data("sensor_temp_003_F.csv", "temperature", "temp_003", 10) # Simulate Fahrenheit output
raw_data_files.append("sensor_temp_003_F.csv")

# --- Data Processing Function (includes comprehensions, lambda, and file ops) ---
def process_sensor_file(filepath):
    print(f"Processing file: {filepath}...")
    sensor_readings = []
    sensor_type = ""
    sensor_id = ""
    target_unit = "C" # Default for temperature, can be overridden by config/file name

    with open(filepath, 'r') as csvfile:
        reader = csv.DictReader(csvfile)
        # List Comprehension: Reads and performs an initial filter/transform on data.
        # The 'if float(row['value']) > 0' acts as a filter on the raw data.
        sensor_readings = [
            {k: (float(v) if k == 'value' else v) for k, v in row.items()} # Dictionary Comprehension here as well
            for row in reader
            if float(row['value']) > 0 # Example filter
        ]

    if sensor_readings:
        sensor_type = sensor_readings[0]['sensor_type']
        sensor_id = sensor_readings[0]['sensor_id']
        if "F" in filepath: # Simple heuristic to determine if target unit is Fahrenheit
            target_unit = "F"
    else:
        print(f"No valid readings found in {filepath}. Skipping.")
        return

    # Use Factory to create appropriate processor instance
    factory = SensorProcessorFactory()
    processor = factory.create_processor(sensor_type, sensor_id, target_unit)

    for reading in sensor_readings:
        processor.process_reading(reading['value'])

    # Dictionary Comprehension: Creates a summary of processed data
    summary = {
        "sensor_id": processor.sensor_id,
        "type": sensor_type,
        "total_readings": len(processor.processed_readings),
        "min_value": min(processor.processed_readings),
        "max_value": max(processor.processed_readings),
        "unit": processor.unit
    }
    print(f"Processed {len(sensor_readings)} readings for {sensor_id}. Summary: {summary}")
    processor.generate_report() # Generates JSON report

    # Archive the raw data file using shutil (File Operations)
    shutil.move(filepath, os.path.join(ARCHIVE_DIR, os.path.basename(filepath)))
    print(f"Archived raw data file: {filepath} -> {ARCHIVE_DIR}")
    return summary

# --- 5. Concurrency: Threading for I/O-bound File Processing ---
# Threading is suitable here because file I/O operations (reading CSV, writing JSON)
# can be slow, and threads allow the program to work on other files while waiting.
print("\n--- Starting Threaded File Processing ---")
threads = []
results = []
for file_path in raw_data_files:
    # Each thread processes one sensor data file concurrently.
    t = threading.Thread(target=lambda p=file_path: results.append(process_sensor_file(p)))
    threads.append(t)
    t.start()

for t in threads:
    t.join() # Wait for all threads to complete
print("--- Threaded File Processing Finished ---")

# --- 6. Concurrency: Multiprocessing for CPU-bound Analysis ---
# Multiprocessing is used for intensive computations that benefit from true parallelism
# across multiple CPU cores, like complex aggregations or statistical models.
def intensive_analysis(data_chunk):
    pid = os.getpid()
    print(f"Process {pid}: Starting intensive analysis on a chunk of {len(data_chunk)} items.")
    time.sleep(1) # Simulate CPU intensive work
    # Example CPU-bound operation: complex aggregation or statistical model fitting
    total_values = sum([item['total_readings'] for item in data_chunk])
    avg_min_value = sum([item['min_value'] for item in data_chunk]) / len(data_chunk) if data_chunk else 0
    print(f"Process {pid}: Finished analysis. Total readings in chunk: {total_values}")
    return {"pid": pid, "total_readings_chunk": total_values, "avg_min_value_chunk": avg_min_value}

print("\n--- Starting Multiprocessed Intensive Analysis ---")
# Filter out None results if any file processing failed
valid_results = [res for res in results if res is not None]

if valid_results:
    # Divide results into chunks for multiprocessing
    chunk_size = max(1, len(valid_results) // multiprocessing.cpu_count())
    chunks = [valid_results[i:i + chunk_size] for i in range(0, len(valid_results), chunk_size)]

    with multiprocessing.Pool() as pool:
        analysis_outputs = pool.map(intensive_analysis, chunks)

    print("--- Multiprocessed Intensive Analysis Finished ---")
    print("Combined Analysis Outputs:")
    for output in analysis_outputs:
        print(output)
else:
    print("No valid results to perform multiprocessing analysis.")

# --- Cleanup Generated Files and Directories ---
# Cleans up the files and directories created during the demonstration.
if os.path.exists('app_config.json'):
    os.remove('app_config.json')
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
if os.path.exists(ARCHIVE_DIR):
    shutil.rmtree(ARCHIVE_DIR)
print("\nCleaned up generated files and directories.")


ConfigManager: Loaded configuration from app_config.json
Ensured directories 'processed_data_reports' and 'raw_data_archive' exist.
Generated sample CSV data: sensor_temp_001.csv
Generated sample CSV data: sensor_humid_002.csv
Generated sample CSV data: sensor_temp_003_F.csv

--- Starting Threaded File Processing ---
Processing file: sensor_temp_001.csv...
Processing file: sensor_humid_002.csv...
TemperatureProcessor for temp_001 created, unit: C
Processed 15 readings for temp_001. Summary: {'sensor_id': 'temp_001', 'type': 'temperature', 'total_readings': 15, 'min_value': 20.0, 'max_value': 27.0, 'unit': 'C'}
HumidityProcessor for humid_002 created, unit: C
Processed 12 readings for humid_002. Summary: {'sensor_id': 'humid_002', 'type': 'humidity', 'total_readings': 12, 'min_value': 60.0, 'max_value': 62.2, 'unit': 'C'}
Processing file: sensor_temp_003_F.csv...
Report for sensor temp_001 saved to processed_data_reports/temp_001_report.json
Archived raw data file: sensor_temp_001.csv -

In [ ]:
# Advanced File Handling: CSV
import csv

# Write to a CSV file
csv_data = [['Name', 'Age'], ['Alice', 30], ['Bob', 24]]
with open('example.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerows(csv_data)
print("CSV file 'example.csv' created.")

# Read from a CSV file
with open('example.csv', 'r') as file:
    reader = csv.reader(file)
    for row in reader:
        print(row)


# Advanced File Handling: JSON
import json

# Write to a JSON file
json_data = {'name': 'Alice', 'age': 30, 'city': 'New York'}
with open('example.json', 'w') as file:
    json.dump(json_data, file, indent=4)
print("JSON file 'example.json' created.")

# Read from a JSON file
with open('example.json', 'r') as file:
    loaded_data = json.load(file)
    print(loaded_data)


CSV file 'example.csv' created.
['Name', 'Age']
['Alice', '30']
['Bob', '24']
JSON file 'example.json' created.
{'name': 'Alice', 'age': 30, 'city': 'New York'}


In [ ]:
# File Operations: os and shutil
import os
import shutil

# Create a directory
if not os.path.exists('my_directory'):
    os.mkdir('my_directory')
    print("Directory 'my_directory' created.")

# Create a file inside the directory
file_path = os.path.join('my_directory', 'test_file.txt')
with open(file_path, 'w') as f:
    f.write('Hello, world!')
print(f"File '{file_path}' created.")

# Copy a file
shutil.copy(file_path, 'my_directory/test_file_copy.txt')
print("File copied to 'my_directory/test_file_copy.txt'.")

# List contents of a directory
print("Contents of 'my_directory':", os.listdir('my_directory'))

# Move a file (rename)
os.rename(file_path, os.path.join('my_directory', 'renamed_file.txt'))
print("File renamed inside 'my_directory'.")

# Delete a file
os.remove('my_directory/test_file_copy.txt')
print("File 'my_directory/test_file_copy.txt' deleted.")

# Delete a directory (if empty)
# os.rmdir('empty_directory') # Use this if directory is empty

# Delete a directory and its contents
shutil.rmtree('my_directory')
print("Directory 'my_directory' and its contents deleted.")


Directory 'my_directory' created.
File 'my_directory/test_file.txt' created.
File copied to 'my_directory/test_file_copy.txt'.
Contents of 'my_directory': ['test_file_copy.txt', 'test_file.txt']
File renamed inside 'my_directory'.
File 'my_directory/test_file_copy.txt' deleted.
Directory 'my_directory' and its contents deleted.


In [ ]:
# List and Dictionary Comprehensions

# List Comprehension: Create a list of squares
numbers = [1, 2, 3, 4, 5]
squares = [n**2 for n in numbers]
print("List of squares:", squares)

# List Comprehension with condition: Get even numbers
even_numbers = [n for n in numbers if n % 2 == 0]
print("List of even numbers:", even_numbers)

# Dictionary Comprehension: Create a dictionary from a list
keys = ['a', 'b', 'c']
values = [1, 2, 3]
my_dict = {k: v for k, v in zip(keys, values)}
print("Dictionary from lists:", my_dict)

# Dictionary Comprehension: Square values of an existing dictionary
original_dict = {'x': 1, 'y': 2, 'z': 3}
squared_dict = {key: value**2 for key, value in original_dict.items()}
print("Dictionary with squared values:", squared_dict)


List of squares: [1, 4, 9, 16, 25]
List of even numbers: [2, 4]
Dictionary from lists: {'a': 1, 'b': 2, 'c': 3}
Dictionary with squared values: {'x': 1, 'y': 4, 'z': 9}


In [ ]:
# Lambda Functions with map and filter

# map with lambda: Square each number in a list
numbers = [1, 2, 3, 4, 5]
squared_numbers = list(map(lambda x: x**2, numbers))
print("Squared numbers (using map and lambda):", squared_numbers)

# filter with lambda: Get even numbers from a list
even_numbers = list(filter(lambda x: x % 2 == 0, numbers))
print("Even numbers (using filter and lambda):", even_numbers)

# map with lambda: Convert strings to uppercase
words = ['hello', 'world', 'python']
uppercased_words = list(map(lambda s: s.upper(), words))
print("Uppercased words (using map and lambda):", uppercased_words)

# filter with lambda: Filter strings longer than 5 characters
long_words = list(filter(lambda s: len(s) > 5, words))
print("Long words (using filter and lambda):", long_words)


Squared numbers (using map and lambda): [1, 4, 9, 16, 25]
Even numbers (using filter and lambda): [2, 4]
Uppercased words (using map and lambda): ['HELLO', 'WORLD', 'PYTHON']
Long words (using filter and lambda): ['python']


In [ ]:
# Advanced OOP: Class Methods and Abstract Classes

# Class Methods
class MyClass:
    class_variable = "I am a class variable"

    def __init__(self, instance_variable):
        self.instance_variable = instance_variable

    @classmethod
    def from_string(cls, input_string):
        # A class method that acts as an alternative constructor
        parts = input_string.split('-')
        return cls(f"Instance from {parts[0]}")

    @classmethod
    def get_class_variable(cls):
        return cls.class_variable

    def display(self):
        print(f"Instance variable: {self.instance_variable}, Class variable: {self.class_variable}")

# Using the class method
obj1 = MyClass("Regular instance")
obj1.display()

obj2 = MyClass.from_string("custom-data")
obj2.display()

print("Accessing class variable via class method:", MyClass.get_class_variable())


# Abstract Classes
from abc import ABC, abstractmethod

class Shape(ABC): # Inherit from ABC to make it an abstract class
    @abstractmethod
    def area(self):
        pass # Abstract method must be implemented by subclasses

    @abstractmethod
    def perimeter(self):
        pass # Another abstract method

    def display_info(self):
        print("This is a shape.")

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return 3.14 * self.radius * self.radius

    def perimeter(self):
        return 2 * 3.14 * self.radius

class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

    def perimeter(self):
        return 2 * (self.width + self.height)

# Cannot instantiate an abstract class directly
# try:
#     s = Shape()
# except TypeError as e:
#     print(f"Cannot instantiate abstract class: {e}")

circle = Circle(5)
print(f"Circle area: {circle.area()}, Perimeter: {circle.perimeter()}")
circle.display_info()

rectangle = Rectangle(4, 6)
print(f"Rectangle area: {rectangle.area()}, Perimeter: {rectangle.perimeter()}")
rectangle.display_info()


Instance variable: Regular instance, Class variable: I am a class variable
Instance variable: Instance from custom, Class variable: I am a class variable
Accessing class variable via class method: I am a class variable
Circle area: 78.5, Perimeter: 31.400000000000002
This is a shape.
Rectangle area: 24, Perimeter: 20
This is a shape.


In [ ]:
# Design Patterns: Singleton and Factory

# Singleton Pattern: Ensures a class has only one instance
class Singleton:
    _instance = None

    def __new__(cls, *args, **kwargs):
        if not cls._instance:
            # Corrected: object.__new__ only takes 'cls' as an argument
            cls._instance = super(Singleton, cls).__new__(cls)
        return cls._instance

    def __init__(self, value):
        # Only initialize if it's the first instance being created
        if not hasattr(self, '_initialized'):
            self.value = value
            self._initialized = True
        print(f"Singleton instance created/retrieved with value: {self.value}")

# Get the first instance
s1 = Singleton("First Instance")
print(f"s1 value: {s1.value}")

# Try to get another instance, it should return the same one
s2 = Singleton("Second Instance")
print(f"s2 value: {s2.value}") # Will still be 'First Instance' (due to _initialized check)

print(f"Are s1 and s2 the same object? {s1 is s2}")
print(f"Value of s1: {s1.value}")
print(f"Value of s2: {s2.value}")

# Factory Pattern: Creates objects without exposing the instantiation logic to the client
class Dog:
    def speak(self):
        return "Woof!"

class Cat:
    def speak(self):
        return "Meow!"

class AnimalFactory:
    def create_animal(self, animal_type):
        if animal_type == 'dog':
            return Dog()
        elif animal_type == 'cat':
            return Cat()
        else:
            raise ValueError("Invalid animal type")

# Using the factory
factory = AnimalFactory()

dog = factory.create_animal('dog')
print(f"Dog says: {dog.speak()}")

cat = factory.create_animal('cat')
print(f"Cat says: {cat.speak()}")

# try:
#     bird = factory.create_animal('bird')
# except ValueError as e:
#     print(f"Error: {e}")


Singleton instance created/retrieved with value: First Instance
s1 value: First Instance
Singleton instance created/retrieved with value: First Instance
s2 value: First Instance
Are s1 and s2 the same object? True
Value of s1: First Instance
Value of s2: First Instance
Dog says: Woof!
Cat says: Meow!


In [ ]:
# Concurrency: Threading and Multiprocessing
import threading
import multiprocessing
import time
import os

# Threading Example (I/O-bound task simulation)
def task_thread(name):
    print(f"Thread {name}: Starting...")
    time.sleep(2)  # Simulate an I/O operation (e.g., network request, file read)
    print(f"Thread {name}: Finishing...")

print("\n--- Threading Example ---")
start_time_threads = time.time()
threads = []
for i in range(3):
    t = threading.Thread(target=task_thread, args=(f'T{i}',))
    threads.append(t)
    t.start()

for t in threads:
    t.join() # Wait for all threads to complete
end_time_threads = time.time()
print(f"All threads finished in {end_time_threads - start_time_threads:.2f} seconds.")


# Multiprocessing Example (CPU-bound task simulation)
def task_process(name, iterations):
    print(f"Process {name}: Starting on PID {os.getpid()}...")
    result = 0
    for _ in range(iterations):
        result += sum(range(1000))
    print(f"Process {name}: Finishing with result (sum of many sums): {result}")

print("\n--- Multiprocessing Example ---")
start_time_processes = time.time()
processes = []
for i in range(3):
    p = multiprocessing.Process(target=task_process, args=(f'P{i}', 10000))
    processes.append(p)
    p.start()

for p in processes:
    p.join() # Wait for all processes to complete
end_time_processes = time.time()
print(f"All processes finished in {end_time_processes - start_time_processes:.2f} seconds.")



--- Threading Example ---
Thread T0: Starting...
Thread T1: Starting...
Thread T2: Starting...
Thread T0: Finishing...
Thread T2: Finishing...
Thread T1: Finishing...
All threads finished in 2.01 seconds.

--- Multiprocessing Example ---
Process P0: Starting on PID 1784...Process P2: Starting on PID 1788...

Process P1: Starting on PID 1785...
Process P1: Finishing with result (sum of many sums): 4995000000
Process P2: Finishing with result (sum of many sums): 4995000000
Process P0: Finishing with result (sum of many sums): 4995000000
All processes finished in 2.34 seconds.
